# Memory Buffer for Streaming Video-LLM Systems

**Assignment** — Hierarchical memory buffer that watches a video
stream, stores useful visual context under a fixed budget, retrieves relevant
past evidence for a text query, and formats that evidence for downstream reasoning.


Let's start by discussing my approach and architecture in general.

### Approach

I would want to start with outlining the design options that were possible:
- flat recent-frame buffer, which is simple but forgets older context (although proven to be quite a strong baseline on current streaming video benchmarks [[1]](https://arxiv.org/abs/2604.02317))
- text-only memory, which is easy to query but throws away visual detail
- full multimodal system, which is powerful and probably the best but too expensive under the limitations of this assignment 

Therefore, I chose a lightweight hierarchical visual memory, inspired by Fluxmem idea [[2]](https://arxiv.org/abs/2603.02096): recent windows are stored densely, older content is compressed into episodes and events, retrieval is coarse-to-fine (inspired by VideoTree [[4]](https://arxiv.org/abs/2405.19209)), and text summaries are used as an helper signal for retreival and also additional information for LLM. This fits the assignment well because it keeps memory bounded, stays query-based, remains visual-first, and can be implemented cleanly in a notebook with small models.

### Architecture

The system follows a perception -> memory -> reasoning design [[5]](https://arxiv.org/abs/2412.09596). A video stream is first converted into short local windows, and each window is mapped to a compact visual embedding using a lightweight encoder. These embeddings are written into a hierarchical memory buffer with three levels [[2]](https://arxiv.org/abs/2603.02096): recent memory for dense short-term storage, episodic memory for merged coherent actions, and long-term event memory for compressed higher-level context.

When a text query arrives, it is encoded into the same retrieval space and used for coarse-to-fine retrieval [[4]](https://arxiv.org/abs/2405.19209): first over recent windows (always included), then over long-term events, then over episodic memory inside the selected regions, with final grounding from episode member windows. Text summaries are included as a supporting signal for retrieval, while the main memory representation remains visual.

The retrieved evidence is then formatted into a readable block that shows how it would be passed to an LLM for downstream reasoning. This final step was the most time-consuming part for me, because I spent a lot of time thinking about the best way to connect retrieved visual information to an LLM. In modern VLMs, this is usually done with a learned bridge such as an MLP projector that maps vision features into the language model space. Since I do not have access to such a pretrained bridge, and training one is outside the scope of this assignment, I had to simplify this part of the pipeline.

My first idea was to use lightweight visual embeddings only for retrieval, and then pass the retrieved frames to a stronger VLM for reasoning. That would still be a reasonable visual-first design. However, the assignment explicitly asks to show how retrieved information would be passed to an LLM. Under that constraint, my final choice was to use visual embeddings for retrieval and use episode/event summaries as the LLM-facing input. This is a simplification, but I think it is the most practical and honest solution here.

Detailed implementation choices (like model selection and hyperparameters, etc.) are described later in the notebook.

Here is my architecture sketch:

![Architecture](architecture.svg)

---
## 0. Setup


## References

[1] [A Simple Baseline for Streaming Video Understanding](https://arxiv.org/abs/2604.02317)

[2] [FluxMem: Adaptive Hierarchical Memory for Streaming Video Understanding](https://arxiv.org/abs/2603.02096)

[3] [FlexMem: Scaling the Long Video Understanding of Multimodal Large Language Models via Visual Memory Mechanism](https://arxiv.org/abs/2603.29252)

[4] [VideoTree: Adaptive Tree-based Video Representation for LLM Reasoning on Long Videos](https://arxiv.org/abs/2405.19209)

[5] [InternLM-XComposer2.5-OmniLive: A Comprehensive Multimodal System for Long-term Streaming Video and Audio Interactions](https://arxiv.org/abs/2412.09596)

[6] [StreamBridge: Turning Your Offline Video Large Language Model into a Proactive Streaming Assistant](https://arxiv.org/abs/2505.05467)